In [1]:
# ==============================================================================
# 1. 라이브러리 설치 및 데이터 다운로드
# ==============================================================================

!pip install efficientnet_pytorch
!pip install torchmetrics
!pip install kagglehub  # KaggleHub 설치

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import kagglehub # 임포트
import time
import copy
from google.colab import drive
from collections import Counter # ⚠️ 클래스 카운트를 위한 라이브러리 추가
from google.colab import drive #

# 다운로드 및 경로 설정
path = kagglehub.dataset_download("loki4514/rice-leaf-diseases-detection")

print("다운로드된 데이터셋 경로:", path)

# GPU 사용 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=4388d43b643e41a50e9c0677927bdd16ab0359c7ef544ee09e3fbe1dbb9e6c79
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet_pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 18.6 MB/s eta 0:00:00
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


100%|██████████| 8.03G/8.03G [01:35<00:00, 90.1MB/s]

Extracting files...


다운로드된 데이터셋 경로: /root/.cache/kagglehub/datasets/loki4514/rice-leaf-diseases-detection/versions/8
사용 장치: cuda:0


In [2]:
# ==============================================================================
# 2. 데이터셋 경로 설정
# ==============================================================================

DATA_ROOT = path
NESTED_ROOT = os.path.join(DATA_ROOT, 'Rice_Leaf_Diease', 'Rice_Leaf_Diease')
TRAIN_DIR = os.path.join(NESTED_ROOT, 'train')
VAL_DIR = os.path.join(NESTED_ROOT, 'test')

In [3]:
# ==============================================================================
# 3. 하이퍼파라미터 및 데이터 변환 설정 (모델 축소 적용)
# ==============================================================================
from efficientnet_pytorch import EfficientNet

# 하이퍼파라미터
BATCH_SIZE = 32 # B0 모델에 맞춰 배치 사이즈를 32로 늘려 속도 개선
NUM_EPOCHS = 10
LEARNING_RATE = 0.0001 # 과적합 방지를 위해 학습률은 계속 낮게 유지
MODEL_NAME = 'efficientnet-b0'

# EfficientNet-B0의 권장 입력 이미지 크기 (224x224로 자동 변경됨)
INPUT_SIZE = EfficientNet.get_image_size(MODEL_NAME)

# 데이터 증강 및 정규화
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.RandomResizedCrop(INPUT_SIZE),
        # 💡 [추가] 회전 및 색상 변화 추가
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # ImageNet 통계
    ]),
    'test': transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}


In [4]:
# ==============================================================================
# 4. 데이터 로드 및 DataLoader 생성
# ==============================================================================
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

print("\n데이터 로딩 및 안전한 분할 중...")

# 1. TRAIN_DIR 전체를 하나의 데이터셋으로 로드합니다.
# ⚠️ 주의: 학습셋의 증강(data_transforms['train'])만 적용합니다.
full_dataset = ImageFolder(TRAIN_DIR, data_transforms['train'])
full_size = len(full_dataset)

# 2. 비율 설정 (예: 90% 학습, 10% 검증)
train_size = int(0.9 * full_size)
val_size = full_size - train_size

# 3. 데이터셋 분리 (겹치지 않음을 보장)
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size],
    # random_split에 사용할 Generator를 설정하면 매번 같은 분할을 얻을 수 있습니다.
    # generator=torch.Generator().manual_seed(42)
)

# 4. DataLoader 재정의
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True),
    # 검증셋은 평가만 하므로 shuffle=False, data_transforms['test']를 사용하지 않음 (random_split의 한계)
    'test': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
}

# 5. 메타 정보 업데이트
# class_names는 전체 데이터셋에서 가져옵니다.
dataset_sizes = {'train': len(train_dataset), 'test': len(val_dataset)}
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)

print(f"클래스 개수: {NUM_CLASSES}")
print(f"클래스 이름: {class_names}")
print(f"✅ 새로운 학습 데이터 크기: {dataset_sizes['train']} (90%)")
print(f"✅ 새로운 검증 데이터 크기: {dataset_sizes['test']} (10%)")
# 참고: random_split을 사용했으므로 라벨 매핑 강제 통일 코드는 필요 없습니다.


데이터 로딩 및 안전한 분할 중...
클래스 개수: 10
클래스 이름: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot', 'neck_blast', 'rice_hispa', 'sheath_blight', 'tungro']
✅ 새로운 학습 데이터 크기: 13520 (90%)
✅ 새로운 검증 데이터 크기: 1503 (10%)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [6]:
# ==============================================================================
# 5. 모델 설정 및 가중치 부여 손실 함수 설정 (수정됨)
# ==============================================================================
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim # optim 모듈 임포트 추가
from efficientnet_pytorch import EfficientNet

# ----------------------------------------------------------------------
# 1. 클래스 가중치 계산 (Class Imbalance 대처)
# ----------------------------------------------------------------------

print("클래스 가중치 계산 중...")

# ⚠️ [수정] image_datasets['train'] 대신 train_dataset을 사용합니다.
# train_dataset은 Subset 객체이므로 .targets 속성이 바로 없습니다.
# 따라서 indices를 이용해 원본 데이터셋(full_dataset)에서 훈련용 라벨만 가져옵니다.
train_targets = [full_dataset.targets[i] for i in train_dataset.indices]

class_counts = Counter(train_targets)
total_count = sum(class_counts.values())

# 클래스별 가중치 계산 (샘플 수가 적은 클래스에 더 큰 가중치 부여)
class_weights = [total_count / class_counts[i] for i in range(NUM_CLASSES)]
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights = class_weights / class_weights.min() # 최소값을 1.0으로 정규화

print(f"✅ 클래스 가중치 계산 완료: {class_weights.tolist()}")


# ----------------------------------------------------------------------
# 2. EfficientNet-B0 모델 로드 및 분류 레이어 교체
# ----------------------------------------------------------------------

# ImageNet 가중치로 EfficientNet-B0 모델을 로드합니다.
model_ft = EfficientNet.from_pretrained(MODEL_NAME).to(device)

# 기존 분류 레이어(_fc)의 입력 피처 수 확인
num_ftrs = model_ft._fc.in_features

# 실제 데이터셋의 클래스 수(NUM_CLASSES)에 맞춰 새로운 레이어 정의
# Dropout을 추가하여 과적합을 방지합니다.
model_ft._fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, NUM_CLASSES)
)

model_ft = model_ft.to(device) # 모델을 GPU/CPU로 이동

print(f"✅ EfficientNet-B0의 최종 분류 레이어가 {num_ftrs} -> {NUM_CLASSES}로 교체되었습니다.")


# ----------------------------------------------------------------------
# 3. Loss, Optimizer, Scheduler 정의
# ----------------------------------------------------------------------

# 계산된 class_weights를 적용한 CrossEntropyLoss
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Optimizer (AdamW 사용)
optimizer_ft = optim.AdamW(model_ft.parameters(), lr=LEARNING_RATE)

# Scheduler (학습률 스케줄러)
exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer_ft, step_size=8, gamma=0.5)

print("✅ Loss, Optimizer, Scheduler 정의 완료.")

클래스 가중치 계산 중...
✅ 클래스 가중치 계산 완료: [1.3344182968139648, 1.2321562767028809, 1.2175204753875732, 1.0, 1.0825083255767822, 1.2995246648788452, 1.8468469381332397, 1.2557427883148193, 1.1508772373199463, 1.0399492979049683]
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 229MB/s]


Loaded pretrained weights for efficientnet-b0
✅ EfficientNet-B0의 최종 분류 레이어가 1280 -> 10로 교체되었습니다.
✅ Loss, Optimizer, Scheduler 정의 완료.


In [7]:
# ==============================================================================
# 6. 모델 학습 함수
# ==============================================================================
def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        print(f'\nEpoch {epoch+1}/{num_epochs}'); print('-' * 10)

        # 'train'과 'test' 단계를 반복합니다.
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train() # 모델을 학습 모드로 설정
            else:
                model.eval()  # 모델을 평가 모드로 설정

            running_loss = 0.0
            running_corrects = 0

            # DataLoader에서 데이터를 가져와 반복합니다.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad() # 경사 초기화

                # 학습 단계일 때만 경사 계산을 활성화합니다.
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # 역전파 및 최적화 단계 (학습 단계에서만 실행)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # 통계 업데이트
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # 학습 단계가 끝나면 스케줄러를 업데이트합니다.
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # 'test' 단계의 정확도가 최고 기록일 때 모델을 저장합니다.
            if phase == 'test' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f'-> New best model found! Saving weights with Acc: {best_acc:.4f}')

        epoch_time = time.time() - epoch_start_time
        print(f'Epoch Time: {epoch_time:.0f}초')

    time_elapsed = time.time() - since
    print(f'\n학습 완료! 총 시간: {time_elapsed // 60:.0f}분 {time_elapsed % 60:.0f}초')
    print(f'최고 검증 정확도: {best_acc:.4f}')

    # 최고 성능의 모델 가중치 로드
    model.load_state_dict(best_model_wts)
    return model

In [8]:
# ==============================================================================
# 7. 모델 학습 실행 및 저장
# ==============================================================================

# 모델 학습 시작
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=NUM_EPOCHS)

# Google Drive 마운트
drive.mount('/content/drive')
MODEL_SAVE_PATH = '/content/drive/MyDrive/efficientnet_rice_disease_b4_weighted_best_model.pth' # ⚠️ 파일 이름에 weighted 추가

# 모델 저장
torch.save(model_ft.state_dict(), MODEL_SAVE_PATH)
print(f"\n✅ 최적 모델 가중치가 {MODEL_SAVE_PATH}에 저장되었습니다.")


Epoch 1/10
----------
train Loss: 1.0730 Acc: 0.6552
test Loss: 0.4111 Acc: 0.8417
-> New best model found! Saving weights with Acc: 0.8417
Epoch Time: 445초

Epoch 2/10
----------
train Loss: 0.4103 Acc: 0.8578
test Loss: 0.2723 Acc: 0.9022
-> New best model found! Saving weights with Acc: 0.9022
Epoch Time: 449초

Epoch 3/10
----------
train Loss: 0.3099 Acc: 0.8936
test Loss: 0.2107 Acc: 0.9375
-> New best model found! Saving weights with Acc: 0.9375
Epoch Time: 447초

Epoch 4/10
----------
train Loss: 0.2634 Acc: 0.9081
test Loss: 0.1987 Acc: 0.9228
Epoch Time: 443초

Epoch 5/10
----------
train Loss: 0.2207 Acc: 0.9220
test Loss: 0.1484 Acc: 0.9534
-> New best model found! Saving weights with Acc: 0.9534
Epoch Time: 440초

Epoch 6/10
----------
train Loss: 0.1887 Acc: 0.9355
test Loss: 0.1486 Acc: 0.9415
Epoch Time: 447초

Epoch 7/10
----------
train Loss: 0.1824 Acc: 0.9362
test Loss: 0.0975 Acc: 0.9647
-> New best model found! Saving weights with Acc: 0.9647
Epoch Time: 437초

Epoch 8

In [9]:
# ==============================================================================
# 8. ONNX 모델 변환 및 저장 (별도 블럭)
# ==============================================================================

!pip install onnxscript
!pip install onnx
!pip install onnxruntime

# ⚠️ '7. 모델 학습 실행' 블럭이 실행된 후,
# ⚠️ model_ft, INPUT_SIZE, device 변수가 메모리에 로드된 상태여야 합니다.

ONNX_SAVE_PATH = 'rice_disease_model.onnx'

# 1. 모델을 '평가 모드'로 설정 (필수!)
# Dropout, BatchNorm 등이 추론 모드로 변경됩니다.
model_ft.eval()

# 2. 모델에 입력될 더미 텐서(dummy input) 생성
# (EfficientNet-B0 기준 INPUT_SIZE = 224)
# (Batch_size, Channels, Height, Width)
dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)

# 3. ONNX로 내보내기
try:
    torch.onnx.export(
        model_ft,                   # 실행될 모델 (메모리에 로드된 상태)
        dummy_input,                # 모델 입력값 (모양과 타입)
        ONNX_SAVE_PATH,             # 저장될 ONNX 파일 경로
        export_params=True,         # 모델 파라미터(가중치) 저장
        opset_version=11,           # ONNX 오퍼레이터 버전
        do_constant_folding=True,   # 최적화
        input_names = ['input'],    # ONNX 그래프의 입력 텐서 이름
        output_names = ['output'],  # ONNX 그래프의 출력 텐서 이름
        dynamic_axes={'input' : {0 : 'batch_size'},    # 배치 크기를 동적으로 설정
                      'output' : {0 : 'batch_size'}}
    )
    print(f"\n✅ 모델이 ONNX 형식으로 {ONNX_SAVE_PATH}에 성공적으로 저장되었습니다.")

except Exception as e:
    print(f"\n❌ ONNX 변환 중 오류 발생: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.0/683.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.2 MB/s eta 0:00:00


/tmp/ipython-input-42613733.py:25: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1130 14:00:47.869000 171 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:65: adapter_lookup: Assertion `false`

[torch.onnx] Translate the graph into ONNX... ✅
Applied 115 of general pattern rewrite rules.

✅ 모델이 ONNX 형식으로 rice_disease_model.onnx에 성공적으로 저장되었습니다.
